# Определение переворота текстовой строки на 180°

Итоговое решение — компактная **PP-LCNet_x1_0_textline_ori**, дообученная на синтетических русских и английских строках. Для каждого бокса возвращается вероятность `p_180`, что он повёрнут вверх ногами.

**Результат на тесте организатора: 1 − Brier = 0.90386951; Brier = 0.09613049.** Отправлен `submission.csv` из корня репозитория с исходными вероятностями, без округления до 0/1. Тестовые метки локально недоступны: результат сообщён организатором после отправки.

Ниже объясняю выбор модели, подготовку данных, валидацию и получение CSV. По умолчанию Run All читает лишь небольшие сохранённые отчёты: **не скачивает данные, не сканирует картинки, не запускает обучение или GPU**. Для предсказаний нужно явно включить `RUN_PREDICTION`.


## 1. Почему готовый классификатор, а не полный OCR

На вход уже даны боксы с текстом. Искать текст и распознавать каждое слово необязательно: нужен только ответ **0° или 180°**. OCR на обеих ориентациях был возможной гипотезой, но добавлял бы вычисления и зависимость от качества распознавания. Он не вошёл в итоговое решение.

Начал с готовой PP-LCNet из PaddleOCR: она уже решает нужную задачу и компактна, что важно при миллионах боксов. Поэтому не пришлось заменять голову OCR на классификационную: у **textline_ori** уже есть категории `0_degree` и `180_degree`. Это не четырёхклассовая модель ориентации документа.

Сначала проверил исходные веса, затем дообучил ту же архитектуру, чтобы сравнение не смешивало одновременно замену модели и изменение данных. Полный OCR, LLM/VLM и внешние inference API в финальном предсказании не используются.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import subprocess
import sys

# Ищем корень по артефакту проекта, без личных путей автора.
candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidates if (p / 'artifacts/model_card.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Откройте ноутбук внутри клонированного репозитория.')
MODEL_CARD = json.loads((ROOT / 'artifacts/model_card.json').read_text(encoding='utf-8'))
VALIDATION = json.loads((ROOT / 'reports/validation/validation_metrics.json').read_text(encoding='utf-8'))
MODEL_DIR = ROOT / 'artifacts/model'
assert MODEL_CARD['model_name'] == 'PP-LCNet_x1_0_textline_ori'
print('Модель:', MODEL_CARD['model_name'])
print('Версия:', MODEL_CARD['version'])
print('Python:', sys.version.split()[0])


## 2. Короткий EDA: что он дал и чего не доказал

Изучил размеры 20 000 тестовых боксов, чтобы понять исходный формат. Надписи отличаются разрешением и качеством; в примерах встречаются русский, английский и цифры. Ниже — сохранённая статистика размеров, без ручной разметки ориентации.

| Признак, px | Q25 | Медиана | Q75 | Максимум |
| --- | ---: | ---: | ---: | ---: |
| Ширина | 129 | 238 | 479.25 | 1599 |
| Высота | 27 | 42 | 76 | 492 |

238 и 42 — отдельные медианы двух признаков, а не обязательно размер одной существующей картинки. Длинный правый хвост показывает, почему среднее плохо описывает типичный бокс.

![Размеры тестовых боксов и соотношение сторон](../docs/figures/test_sizes.png)

Графики ограничены 99-м перцентилем **только для визуализации**: изображения из хвоста не исключались из тестового предсказания.

EDA помог рассмотреть сохранение пропорций для собственного CNN, в том числе вариант 48×384. Но он **не использовался** в финальном fine-tune. Итоговый resize 160×80 взят из официального рецепта PP-LCNet, не оптимизирован по этим гистограммам и не сравнивался с альтернативами в отдельной абляции. Геометрическое покрытие не равно качеству классификации.


In [ ]:
# Выключено, чтобы отчёт работал без тестовых PNG.
RUN_EDA = False
EDA_IMAGES_DIR = ROOT / 'data/test/images'

if RUN_EDA:
    from PIL import Image
    import numpy as np

    with (ROOT / 'sample_submission.csv').open(encoding='utf-8', newline='') as stream:
        image_ids = [row['image_id'] for row in csv.DictReader(stream)]
    sizes = []
    for image_id in image_ids:
        filename = image_id if Path(image_id).suffix else image_id + '.png'
        # Читаем заголовок PIL; массивы пикселей не загружаются и не накапливаются.
        with Image.open(EDA_IMAGES_DIR / filename) as image:
            sizes.append(image.size)  # (width, height), не (height, width)

    print('Изображений:', len(sizes))
    for name, axis in [('width', 0), ('height', 1)]:
        values = [size[axis] for size in sizes]
        quantiles = np.percentile(values, [25, 50, 75, 100])
        print(name, dict(zip(['Q25', 'median', 'Q75', 'max'], quantiles)))
else:
    print('Повторный EDA выключен; выше приведены сохранённые результаты.')


## 3. Сначала baseline, затем адаптация

Готовые модели проверены на одной и той же парной синтетической validation. Сравниваю `1 − Brier`, а не только accuracy, потому что отправляется вероятность.

| Эксперимент | Синтетическая validation: 1 − Brier | Тест организатора: 1 − Brier |
| --- | ---: | ---: |
| PP-LCNet x0.25, исходные веса | 0.89549513 | — |
| PP-LCNet x1.0, исходные веса | 0.89493120 | 0.85088864 |
| **PP-LCNet x1.0, fine-tune RU/EN** | **0.99542934** | **0.90386951** |
| Fine-tune + жёсткое округление по 0.5 | 0.99370005 | Не оценён |

x1.0 **не выиграла** у x0.25 по синтетическому holdout: результаты почти одинаковы, x0.25 немного выше. x1.0 стала проверенным тестовым baseline, поэтому продолжил с ней для сопоставления до/после адаптации. Для x0.25 тестового результата нет. ResNet-18 рассматривался, но завершённого сравнения не было.

Тестовые оценки сообщены автором по ответу организатора. Из `sample_submission.csv` нельзя вычислить тестовый Brier: значения в нём не являются истинными метками. Числа исходных моделей и срезы сохранены в `reports/baseline_metrics.json`, fine-tune — в `reports/validation/`.


### Почему решил дообучать

Общий балл скрывал неодинаковое качество по языкам. Срезы готовой x1.0 дали наблюдаемую мотивацию для адаптации:

| Язык | Парных validation-примеров | Brier: исходные веса | Brier: fine-tune |
| --- | ---: | ---: | ---: |
| Английский | 10 734 | 0.056076 | 0.003854 |
| Русский | 10 536 | 0.154982 | 0.005301 |

Особенно слабой была группа RU с буквами и цифрами: **1556 примеров, Brier = 0.320837** до fine-tune. Буквы, цифры и смешанные строки оставлены в данных; `content_type` используется для анализа, а не удаления групп.

**Гипотеза:** RU/EN fine-tune улучшит признаки, полезные для определения ориентации этих строк. Это не то же самое, что «понимание языка» или распознавание текста. Также результат не доказывает, что исходная модель обучалась без русского: её исходный состав train не проверялся. Гипотеза проверена сравнением той же архитектуры до/после обучения.


## 4. Данные и честная валидация

Использованы первые RU/EN шарды `nvidia/OCR-Synthetic-Multilingual-v1`. Из синтетических страниц извлечены `line_bboxes`: по 40 000 кропов на язык, не более трёх строк с одной страницы, seed 42. После геометрического фильтра `width / height >= 1` осталось 71 155 изображений. Этот фильтр отбирает форму бокса, но сам по себе не доказывает направление текста.

Данные разделены **до** создания перевёрнутых вариантов: 60 520 исходных изображений в train и 10 635 в validation. Один `GroupShuffleSplit` по исходной странице, отдельно для каждого языка, `validation_size=0.15`, seed 42. Страницы не пересекаются между train и validation; это один holdout, не k-fold CV.

Каждый исходный кроп даёт два примера: исходный с меткой 0 и повёрнутый на 180° с меткой 1. Получаются 121 040 train-примеров и 21 270 validation-примеров с балансом классов 50/50. Перемешивание воспроизводимо, seed 42 для train и 43 для validation. Связанные ориентации одной строки всегда остаются в одной части. Исходные тестовые изображения не использованы для обучения или ручной разметки.

Источник содержит синтетические страницы с аннотациями, не реальные фотографии вывесок. В синтетической генерации известна нормальная ориентация, поэтому исходному кропу назначается `y=0`, а его повороту — `y=1`, без ручной разметки. Создание двух ориентаций не удваивает разнообразие исходных надписей: это два связанных примера одного текста. Нечитаемый или симметричный фрагмент может оставаться визуально неоднозначным.

Групповой split важен не только для двух ориентаций: кропы одной страницы разделяют фон и оформление. Он ограничивает утечку страницы, но не исключает сходство шрифтов или шаблонов разных синтетических страниц. Размер train — выбранный бюджет подготовки, а не правило, будто train обязан быть в четыре раза больше теста. Баланс 50/50 задаётся созданием пар, а не предположением о неизвестном балансе теста.


### Где сохранённые данные

Данные не включены в Git. Автор размещает архив в [папке Google Drive](https://drive.google.com/drive/folders/1G5tqSNlCG89dKWfuCzvM4TkXhgPvkCPc?usp=sharing). **На момент обновления документации загрузка архива и доступ по ссылке не подтверждены.** Ожидаемые каталоги и назначение raw/processed описаны в [docs/data.md](../docs/data.md).

Для точного повторения обучающего набора предпочтительны сохранённые `data/processed/line_crops/{ru,en}/` с `train.csv` и `validation.csv`. Raw H5 нужны только для повторного извлечения. Ревизия `69696a1cc543ef3a0f8e9892a89c17293e915263` восстановлена из локальных download metadata обоих шардов и закреплена в `configs/data.yaml`.

Для **получения submission** train и raw не нужны: достаточно тестовых картинок, `sample_submission.csv` и сохранённого в Git экспорта модели.


## 5. Что именно дообучалось

Старт — официальные двухклассовые веса **textline_ori**, не четырёхклассовая document-orientation модель. Архитектура и бинарная голова 0°/180° сохранены.

Параметры завершённого эксперимента: 10 эпох, batch size 64, learning rate 0.05, warmup 1 эпоха, Momentum 0.9, cosine learning-rate schedule, L2 `1e-5`, seed 42. Использованы `RandAugment` (`rand-m9-mstd0.5-inc1`) и `RandomErasing`, loss — cross-entropy. Лучший checkpoint выбран по accuracy синтетической валидации, а не по тестовому баллу; выбран epoch 10.

**Фактический вход модели: ширина 160 × высота 80 px**, тензор `[3, 80, 160]`. `ResizeImage: size: [160, 80]` означает `[W, H]`. Размер 48×384 рассматривался на этапе EDA, но в этом эксперименте не использован. RGB нормализуется с ImageNet mean/std из официального конфигурационного файла. На предсказании применяется preprocessing сохранённого экспорта; дополнительный resize вручную не нужен.

Этот эксперимент нельзя называть обучением «без аугментаций»: RandAugment и RandomErasing входят в фактический рецепт. Brier посчитан по вероятностям отдельно после обучения; checkpoint выбирался по accuracy.

Нормализация переводит RGB-каналы в диапазон 0–1 и применяет `(channel − mean) / std`, где mean `[0.485, 0.456, 0.406]`, std `[0.229, 0.224, 0.225]`. Нормализуется яркость цветовых каналов, **не координаты пикселей**. Эти параметры сохраняются из рецепта предобученной модели. Официальный resize оставлен для сопоставимого эксперимента, а не объявляется оптимальным для любых длин строк.


In [ ]:
# Читаем сохранённый отчёт, не выдавая синтетическую оценку за тестовую.
print('Синтетическая validation:')
for key in ['validation_samples', 'accuracy', 'brier', '1_minus_brier']:
    print(f'  {key}: {VALIDATION[key]}')
test = MODEL_CARD['test']
print('Тест организаторов (сообщённый результат):')
print('  1 − Brier:', test['user_reported_1_minus_brier'])
print('  Brier:', test['user_reported_brier'])
model_files = ['inference.json', 'inference.pdiparams', 'inference.yml']
missing = [name for name in model_files if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Не хватает файлов экспорта: {missing}')
size_mib = sum((MODEL_DIR / name).stat().st_size for name in model_files) / 2**20
print(f'Размер экспорта: {size_mib:.2f} MiB')

## 6. Почему сохраняются вероятности

Метрика `Brier = mean((p_180 − y)**2)` оценивает и направление, и уверенность. При `y=0` прогноз 1 даёт ошибку 1, а прогноз 0.5 — 0.25. Но это не значит, что всем надо ставить 0.5: уверенный правильный прогноз лучше осторожного. Нужны информативные вероятности, а не искусственная уверенность.

Идея `p >= 0.5 → 1`, иначе 0 проверена отдельно. На validation балл снизился с **0.99542934 до 0.99370005**. Для жёстких 0/1 Brier равен доле ошибок, а `1 − Brier` — accuracy. Это объясняет результат, но не позволяет узнать эффект на реальном тесте без отправки. Основной CSV оставлен вероятностным; калибровки на тестовых метках не было.

`topk=2` возвращает обе категории. В `src/predict.py` вероятность 180° выбирается сопоставлением **`label_names` и `scores` текущей картинки**, а не фиксированным индексом отсортированного score. В PaddleX 3.7.2 `class_ids` мог содержать матрицу всего батча; её разворачивание давало неверное соответствие. Это учтено: проверяются обе метки, диапазон и сумма вероятностей.


### Ограничения и скорость

Синтетическая validation дала 0.99542934, реальный тест — 0.90386951. Разрыв согласуется с ограниченным переносом синтетики на реальные фото, но не определяет единственную причину ошибок и не доказывает достижение максимума. Не следует заключать, что достаточно просто увеличить число страниц того же генератора.

Следующие возможные проверки: внешняя реальная validation, короткие слова и логотипы, управляемое размытие/уменьшение разрешения, сохранение пропорций и калибровка на отдельной от оценочной выборке. Они **не заявляются выполненными улучшениями**.

Экспорт занимает **6.53 MiB** (6 849 052 байта; это размер файлов, не число параметров). Один проход по 21 270 синтетическим примерам на RTX A4000 занял 24.07 s, около 884 изображений/s, batch 128. Замер включает чтение и preprocessing, но не инициализацию модели. Это один end-to-end замер, не production-бенчмарк и не стандартизованное сравнение CPU/GPU.


## 7. Как устроен код и что запускать проверяющему

Ноутбук хранит объяснения и управляет запуском. Общие функции вынесены в `src`, чтобы не копировать разные реализации в ячейки:

| Файл | Ответственность |
| --- | --- |
| [src/prepare_data.py](../src/prepare_data.py) | Скачать H5, извлечь строки, фильтровать метаданные, разделить страницы |
| [src/build_orientation_dataset.py](../src/build_orientation_dataset.py) | Создать пары 0°/180° и manifests после готового split |
| [src/train.py](../src/train.py) | Проверка конфигурации, train/evaluate/export; параметры из YAML |
| [src/predict.py](../src/predict.py) | Локальный inference, извлечение вероятности, validation и submission |

Для получения CSV не нужно обучать заново. Установите Python 3.12 и зависимости по [README](../README.md), положите тестовые картинки в `data/test/images`, затем включите предсказание ниже. Модель берётся из `artifacts/model`, preprocessing — из сохранённого экспорта, без внешнего inference API.

Обучение описано отдельно в [docs/training.md](../docs/training.md); нужны данные, разрешённый GPU и закреплённые PaddleX/PaddleClas. Один seed не гарантирует побитовое совпадение нового обучения на другом оборудовании. Run All не устанавливает зависимости и не запускает fine-tune.


In [ ]:
# Единственная конфигурационная ячейка для получения нового CSV.
RUN_PREDICTION = False
RUN_COMPARISON = RUN_PREDICTION  # Можно включить отдельно для уже предсказанного файла.
SAMPLE_PATH = ROOT / 'sample_submission.csv'
IMAGES_DIR = ROOT / 'data/test/images'
OUTPUT_PATH = ROOT / 'outputs/submission_reproduced.csv'
DEVICE = 'cpu'
PHYSICAL_GPU_ID = None  # Для GPU явно укажите карту, которую вам разрешено использовать.
BATCH_SIZE = 128

assert DEVICE in {'cpu', 'gpu'}
assert BATCH_SIZE > 0
if DEVICE == 'gpu':
    assert isinstance(PHYSICAL_GPU_ID, int) and PHYSICAL_GPU_ID >= 0, 'Нужен явный physical GPU ID.'
if OUTPUT_PATH.resolve() == (ROOT / 'submission.csv').resolve():
    raise ValueError('Новый прогон не должен перезаписывать отправленный submission.csv.')


In [ ]:
if RUN_PREDICTION:
    required = [SAMPLE_PATH, IMAGES_DIR, ROOT / 'src/predict.py']
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError('Укажите существующие входные файлы: ' + ', '.join(missing))
    if OUTPUT_PATH.exists():
        raise FileExistsError(f'Выберите новое имя результата: {OUTPUT_PATH}')
    # Настоящий inference: сохранённый submission не копируется вместо расчёта.
    command = [
        sys.executable, str(ROOT / 'src/predict.py'),
        '--model-dir', str(MODEL_DIR), '--device', DEVICE,
        '--batch-size', str(BATCH_SIZE), '--sample', str(SAMPLE_PATH),
        '--images', str(IMAGES_DIR), '--output', str(OUTPUT_PATH),
    ]
    if DEVICE == 'gpu':
        command.extend(['--physical-gpu-id', str(PHYSICAL_GPU_ID)])
    # Новый процесс задаёт CUDA_VISIBLE_DEVICES до импорта Paddle.
    subprocess.run(command, cwd=ROOT, check=True)
    print('Новый CSV:', OUTPUT_PATH)
else:
    print('Предсказание выключено. Настройте пути и RUN_PREDICTION=True.')


## 8. Проверка воспроизводимости: файл и численные отклонения

Исходный CSV получен на RTX A4000, Paddle 3.3.0, batch 128. **Полный повторный GPU-прогон на этой конфигурации воспроизвёл все 20 000 ID, порядок и вероятности побитово**: SHA256 совпал с отправленным файлом (`a1b8f37980b35b76dc4b50d39cf40cfb30624e2cb34fa8fc22bfa9a50be0a41d`). Это проверенный результат, не обещание одинаковой арифметики на любом оборудовании.

Полный CPU-прогон сохранил все ID и классы, но численно отличалась 161 вероятность: максимальное абсолютное отклонение около **2.0027e-5**, среднее **8.10168e-8**, смен класса по 0.5 — **0**. Поэтому одинаковый SHA256 между CPU/GPU не обещается. Сохранённый отчёт — `reports/reproducibility.json`.

Следующая ячейка после явно включённого сравнения читает **реально заново предсказанный** CSV: проверяет схему, порядок ID, диапазон, максимальную и среднюю разницу, классы и SHA256. Она не использует reference для предсказаний и не заменяет расчёт копированием. Разница файлов и численная разница — разные проверки.


In [ ]:
def read_submission(path):
    with path.open(encoding='utf-8', newline='') as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ['image_id', 'p_180']:
            raise ValueError(f'Неверная схема: {path}')
        records = list(reader)
    if any(None in row or any(value is None for value in row.values()) for row in records):
        raise ValueError(f'Ожидалось ровно два значения в каждой строке: {path}')
    ids = [row['image_id'] for row in records]
    probabilities = [float(row['p_180']) for row in records]
    if not ids or len(ids) != len(set(ids)):
        raise ValueError(f'Пустой CSV или повторяющиеся ID: {path}')
    if not all(math.isfinite(p) and 0 <= p <= 1 for p in probabilities):
        raise ValueError(f'Некорректные вероятности: {path}')
    return ids, probabilities

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if RUN_COMPARISON:
    reference_path = ROOT / 'submission.csv'
    reference_ids, reference_p = read_submission(reference_path)
    new_ids, new_p = read_submission(OUTPUT_PATH)
    with SAMPLE_PATH.open(encoding='utf-8', newline='') as stream:
        expected_ids = [row['image_id'] for row in csv.DictReader(stream)]
    if new_ids != reference_ids or new_ids != expected_ids:
        raise ValueError('Количество, ID или порядок не совпадают с sample/reference.')
    if len(new_ids) != 20000:
        raise ValueError('Ожидалось ровно 20 000 тестовых изображений.')
    differences = [abs(left - right) for left, right in zip(new_p, reference_p)]
    class_differences = sum((left >= .5) != (right >= .5)
                            for left, right in zip(new_p, reference_p))
    comparison = {
        'rows': len(new_ids), 'ids_and_order_identical': True,
        'probabilities_changed': sum(diff != 0 for diff in differences),
        'max_absolute_difference': max(differences),
        'mean_absolute_difference': sum(differences) / len(differences),
        'class_differences_at_0_5': class_differences,
        'reference_sha256': sha256_file(reference_path),
        'new_sha256': sha256_file(OUTPUT_PATH),
    }
    comparison['files_identical'] = comparison['reference_sha256'] == comparison['new_sha256']
    print(json.dumps(comparison, ensure_ascii=False, indent=2))
else:
    print('Сравнение выключено. Включите RUN_COMPARISON после нового inference.')

# Маленький JSON можно читать без данных и запуска inference.
saved_report = ROOT / 'reports/reproducibility.json'
if saved_report.is_file():
    saved_checks = json.loads(saved_report.read_text(encoding='utf-8'))
    print('Сохранённая проверка от', saved_checks['checked_on'])
    for device in ['gpu', 'cpu']:
        check = saved_checks[device + '_rerun']
        print(device, 'побитовое совпадение:', check['byte_identical'],
              'максимальная разница p_180:', check['max_abs_probability_difference'])


## Использованные open-source источники

- [NVIDIA OCR-Synthetic-Multilingual-v1](https://huggingface.co/datasets/nvidia/OCR-Synthetic-Multilingual-v1) — исходные синтетические данные, CC BY 4.0; NVIDIA / Ryan Chesler. Использованная HF ревизия `69696a1cc543ef3a0f8e9892a89c17293e915263` восстановлена по локальным download metadata обеих языковых частей.
- [PaddleOCR: text-line orientation classification](https://www.paddleocr.ai/latest/en/version3.x/module_usage/textline_orientation_classification.html) — описание PP-LCNet textline orientation.
- [PaddleX](https://github.com/PaddlePaddle/PaddleX) и [PaddleClas](https://github.com/PaddlePaddle/PaddleClas) — архитектура, обучение и локальный inference; Apache 2.0.

Подготовка данных, групповой split, проверка вероятностей и формирование submission выполнены в коде этого репозитория. Контрольные суммы весов и исходного submission сохранены в `artifacts/model_card.json`.